# Helsinki-NLP/opus-mt-en-ru: zero-shot → fine-tune

Select a **T4 GPU**. Target **1–2 hours** for the default three-epoch fine-tuning workflow, including zero-shot/final batched evaluation; this is not measured here. OPUS may include Tatoeba: this test set may have appeared during pretraining, so the comparison is not proof of clean generalization. The raw TSV and source-group split are the same as scratch; multi-reference sacreBLEU 13a and chrF2 share the exact scoring implementation. No Kaggle API or data downloader is used.

## 1. Clone/update the implementation branch and define live command streaming
A clean existing checkout is fast-forwarded; local edits are preserved by failing early. stdout/stderr are streamed and failures propagate. After PR merge you may change the ref to `main`.

In [ ]:
from pathlib import Path
import subprocess
import sys

def stream(command):
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        status = process.wait()
    if status:
        raise subprocess.CalledProcessError(status, command)

REPO_URL = "https://github.com/vcdx1111/Machine-translation-English-to-Russian-.git"
REPO_REF = "feat/scratch-transformer"
REPO = Path("/content/Machine-translation-English-to-Russian-")
if not REPO.exists():
    stream(["git", "clone", "--branch", REPO_REF, REPO_URL, str(REPO)])
else:
    assert not subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip(), "Preserve local changes before updating"
    stream(["git", "-C", str(REPO), "fetch", "origin", REPO_REF])
    stream(["git", "-C", str(REPO), "switch", REPO_REF])
    stream(["git", "-C", str(REPO), "merge", "--ff-only", "origin/" + REPO_REF])
%cd /content/Machine-translation-English-to-Russian-
stream(["git", "log", "-1", "--oneline"])

## 2. Install pinned Helsinki dependencies
Use `requirements-helsinki.txt` (includes the scratch requirements). Torchvision is pinned to match torch 2.6 and avoid incompatible Colab preinstalls when Transformers imports it. If installation requires a runtime restart, restart and rerun from the top before importing Transformers. Public model downloads need no personal HF token. CPU development tests use a random local Marian fixture; they are not Helsinki quality tests.

In [ ]:
stream([sys.executable, "-m", "pip", "install", "-r", "requirements-helsinki.txt"])
import torch
assert torch.cuda.is_available(), "Select T4 GPU"
print(torch.__version__, torch.cuda.get_device_name(0))

## 3. Mount Drive and choose a separate Helsinki run
Keep the same run name and `RESUME=True` after reconnect. Zero-shot's resolved model revision, raw-data/reference hashes, and config contract are persisted in `run_manifest.json`; tokenizer and model use the same snapshot. Trainer checkpoints include optimizer/scheduler/scaler/RNG and early-stopping state. Keep the whole run directory; do not copy only model weights for resume.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
CONFIG = "configs/helsinki_colab.yaml"
RUN_NAME = "helsinki_v1_seed42"
RESUME = False
RUN = Path("/content/drive/MyDrive/en_ru_translation") / RUN_NAME
CHECKPOINTS = RUN / "checkpoints"
RESULTS = RUN / "results"
CHECKPOINTS.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
def helsinki_command(stage):
    return [sys.executable,"-u","-m","src.helsinki","--config",CONFIG,"--stage",stage,
            "--checkpoint-dir",str(CHECKPOINTS),"--results-dir",str(RESULTS)]

## 4. Inspect checked-in data and shared split
No test text is used for fine-tuning or tokenizer fitting. Helsinki keeps its pretrained tokenizer. Truncation is a runtime policy and never removes a reference from evaluation. Split identity is recorded for model comparison.

In [ ]:
import yaml, json
from src.data import selected_splits
cfg = yaml.safe_load(Path(CONFIG).read_text())
assert Path(cfg["paths"]["raw"]).is_file()
rows = selected_splits(cfg)
print({k:len(v) for k,v in rows.items()})
print(rows["train"][:2])
del rows

## 5. Zero-shot evaluation BEFORE updates
Load public Helsinki weights, pin their resolved commit in Drive, then evaluate full greedy and full beam 4. Results are written under `results/zero_shot/` with training time zero and an empty train-log header. Existing matching zero-shot output is reused after reconnect; mismatched data/config fails. This stage still uses test labels for reporting only, never selecting a checkpoint.

In [ ]:
stream(helsinki_command("zero-shot"))
zero = json.loads((RESULTS/"zero_shot/metrics.json").read_text())
print(json.dumps(zero,ensure_ascii=False,indent=2))

## 6. Measure epoch one, save and stop before fine-tuned test evaluation
Default: three epochs, AdamW 3e-5, 5% warmup, FP16, train batch 64 × accumulation 2, length grouping, full validation loss, patience 2. Test generation is outside validation. Live Trainer steps, losses and an epoch summary/ETA appear below. The planned three-epoch linear scheduler is retained, so the next cell resumes the SAME schedule.

In [ ]:
if not RESUME:
    stream(helsinki_command("fine-tune") + ["--stop-after-epochs","1","--skip-test"])
else:
    print("Keep the existing checkpoint and timing.")

## 7. Inspect timing and continue
The projection adds two copies of measured zero-shot generation time to the planned training time (zero-shot + final evaluation). Changed output lengths and Drive saves can change it. If OOM occurs use a new run with a smaller batch, optionally increasing accumulation to keep effective batch 128. Do not change optimizer/schedule settings inside an existing manifested run.

In [ ]:
estimate = json.loads((RESULTS/"finetuned/runtime_estimate.json").read_text())
zero = json.loads((RESULTS/"zero_shot/metrics.json").read_text())
print(json.dumps(estimate,indent=2))
projected = estimate["projected_training_hours"] + 2*zero["evaluation_seconds"]/3600
print(f"Projected training + zero-shot/final evaluation: {projected:.2f} h (excludes setup/Drive overhead)")
stream(helsinki_command("fine-tune") + ["--resume","latest"])

## 8. Inspect results and make the comparison table
Fine-tuned results use the best validation-loss checkpoint, with full greedy + full beam 4 and the same references as zero-shot/scratch. Each result directory has `metrics.json`, `train_log.csv`, `samples.txt`. Generated artifacts stay on Drive. The comparison script refuses known held-out mismatches and excludes CPU smoke files by default; older fast-result files without exact test fingerprints are labeled legacy/unverified. Add the scratch results directory to the list after its run finishes.

In [ ]:
for stage in ("zero_shot","finetuned"):
    directory = RESULTS/stage
    print(stage, (directory/"metrics.json").read_text())
    print((directory/"train_log.csv").read_text())
    print((directory/"samples.txt").read_text())
COMPARE_PATHS = [str(RESULTS)]
# Example after large scratch training:
# COMPARE_PATHS.append("/content/drive/MyDrive/en_ru_translation/scratch_large_v3_seed42/results")
stream([sys.executable,"scripts/compare_results.py",*COMPARE_PATHS,"--output",str(RUN/"comparison")])

## 9. Send back
Share both stage result directories, `run_manifest.json` (contains metadata, not credentials), exact YAML/commit, GPU name, first-epoch time, wall time, errors and best/completed epoch. Keep Trainer checkpoints and `final/` on Drive. Report the OPUS/Tatoeba overlap limitation beside every portfolio comparison.